In [ ]:
!pip install ultralytics
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


#Get Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="uSnjklFio9077gTkCl4Q")
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to football-players-detection-1 in yolov11:: 100%|██████████| 1338/1338 [00:02<00:00, 477.57it/s]


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="uSnjklFio9077gTkCl4Q")
project = rf.workspace("label-futsal").project("futsal-bpfx3")
version = project.version(3)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Futsal-3 in yolov11:: 100%|██████████| 3780/3780 [00:00<00:00, 6359.82it/s]


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="uSnjklFio9077gTkCl4Q")
project = rf.workspace("football-rhxb8").project("futsal-iclv9")
version = project.version(1)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to futsal-1 in yolov11:: 100%|██████████| 2256/2256 [00:00<00:00, 6384.80it/s]


In [ ]:
import os
import shutil

#------------------------------------------
# 1) تعريف مسارات كل Dataset
#------------------------------------------

# عدّل المسارات حسب مكان ملفاتك داخل Google Drive أو Colab
dataset1_path = "/content/futsal-1"
dataset2_path = "/content/football-players-detection-1"
dataset3_path = "/content/Futsal-3"

output_path = "/content/merged_dataset"

#------------------------------------------
# 2) دالة لتعديل الـ labels حسب الـ mapping
#------------------------------------------
def fix_labels(dataset_path, mapping):
    for split in ["train", "valid", "test"]:
        labels_dir = os.path.join(dataset_path, split, "labels")
        if not os.path.exists(labels_dir):
            continue

        for file in os.listdir(labels_dir):
            if not file.endswith(".txt"):
                continue

            path = os.path.join(labels_dir, file)
            new_lines = []

            with open(path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    old_id = int(parts[0])

                    # تجاهل الكلاسات اللي مش عايزينها
                    if old_id not in mapping:
                        continue

                    # تعديل رقم الكلاس
                    parts[0] = str(mapping[old_id])
                    new_lines.append(" ".join(parts))

            # كتابة الملف بعد التعديل
            with open(path, "w") as f:
                f.write("\n".join(new_lines))


#------------------------------------------
# 3) تطبيق التعديل على كل Dataset
#------------------------------------------

# Dataset 1
mapping1 = {0: 0, 2: 1}
fix_labels(dataset1_path, mapping1)

# Dataset 2
mapping2 = {0: 0, 2: 1}
fix_labels(dataset2_path, mapping2)

# Dataset 3 (جاهز بالفعل)
mapping3 = {0: 0, 1: 1}
fix_labels(dataset3_path, mapping3)

print("✔ تم تعديل الكلاسات داخل كل Dataset بنجاح")


#------------------------------------------
# 4) دمج الصور واللابيلز داخل فولدر واحد
#------------------------------------------

# إنشاء فولدر الناتج
for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(output_path, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_path, split, "labels"), exist_ok=True)

def merge(dataset_path, output_path):
    for split in ["train", "valid", "test"]:
        img_src = os.path.join(dataset_path, split, "images")
        lbl_src = os.path.join(dataset_path, split, "labels")

        img_dst = os.path.join(output_path, split, "images")
        lbl_dst = os.path.join(output_path, split, "labels")

        if not os.path.exists(img_src):
            continue

        for file in os.listdir(img_src):
            src = os.path.join(img_src, file)
            dst = os.path.join(img_dst, f"{dataset_path.split('/')[-1]}_{file}")
            shutil.copy(src, dst)

        for file in os.listdir(lbl_src):
            src = os.path.join(lbl_src, file)
            dst = os.path.join(lbl_dst, f"{dataset_path.split('/')[-1]}_{file}")
            shutil.copy(src, dst)


# دمج الـ datasets
merge(dataset1_path, output_path)
merge(dataset2_path, output_path)
merge(dataset3_path, output_path)

print("✔ تم دمج الداتا سيت كلها داخل merged_dataset")


#------------------------------------------
# 5) إنشاء ملف data.yaml النهائي
#------------------------------------------

yaml_content = """train: merged_dataset/train/images
val: merged_dataset/valid/images
test: merged_dataset/test/images

nc: 2
names: ['ball', 'player']
"""

with open("/content/merged_dataset/data.yaml", "w") as f:
    f.write(yaml_content)

print("✔ تم إنشاء ملف data.yaml النهائي")
print("🎉 الداتا جاهزة للـ Training!")


✔ تم تعديل الكلاسات داخل كل Dataset بنجاح
✔ تم دمج الداتا سيت كلها داخل merged_dataset
✔ تم إنشاء ملف data.yaml النهائي
🎉 الداتا جاهزة للـ Training!


In [ ]:
!yolo detect train model=yolo11l.pt data=/content/merged_dataset/data.yaml epochs=100 imgsz=640 batch=16 name=ball_player_model


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=ball_player_model3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0,